# SODE-Guard v2 re-run on ICS3D (Colab)

This notebook re-runs SODE-Guard with the three modules from the research report, over **five seeds**, using one released pipeline (`scripts/v2_run.py`).

| Module | What it adds | Models |
|---|---|---|
| 1. Clean macro-F1 | logit-adjusted loss, AdamW + cosine LR, 60 epochs with early stopping on validation macro-F1, post-hoc logit adjustment tuned on validation data | every neural model |
| 2. EOT-matched adversarial training | N-FGSM through the 8-path mean logits with fresh Brownian paths, ε ramp, early stopping on robust validation macro-F1 | `sode_at`, `ode_at`, `mlp_at` (same recipe) |
| 3. Smoothing certificate | Gaussian input noise (a pure-diffusion prefix of the SDE) + consistency regularisation; Cohen et al. certificate over input noise and Brownian paths jointly | `sode_rs`, `mlp_rs` |

Also trained: `sode_base` (the ICLR-rebuttal recipe on the new split, the "before" row), a neural ODE and an MLP with module 1, and LightGBM, XGBoost and a random forest.

**Protocol.** Exact duplicates and conflicting-label flows are removed before splitting. Every seed draws a new stratified 60/20/20 split; validation is split into a selection half (early stopping, post-hoc τ) and a calibration half (held back). Attacks run on a class-stratified test subset and are also reweighted to the test class mix. Results are aggregated as mean ± 95% CI over seeds, with paired-bootstrap contrasts and Holm correction over a family declared in the script.

**Runtime.** Choose *Runtime → Change runtime type → GPU* (A100 or L4 if you can; a T4 works but is slower). The full five-seed run is long, a rough guess is 4–8 hours on a T4 and less than half that on an A100. Everything is written to Google Drive and **every stage resumes where it stopped**, so after a disconnect re-run steps 1–3 and then the stage that was interrupted.

**At the end** (step 12): download `v2_results.zip` and upload it to the chat.

In [ ]:
!nvidia-smi -L || echo "No GPU: switch the runtime to GPU" 

## 1. Code and dependencies

In [ ]:
%cd /content
!rm -rf CV && git clone -q --depth 1 -b claude/implement-ode-guard-A39oM https://github.com/rogerpanel/CV.git
%cd /content/CV/SODE-Guard
!git log -1 --format='code commit: %h %ci'
!pip -q install kaggle kagglehub lightgbm xgboost scikit-learn scipy pandas

## 2. Save results to Google Drive (survives disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/sode_guard_v2'
import os; os.makedirs(OUT, exist_ok=True); print(OUT)
SEEDS = '0 1 2 3 4'

## 3. Kaggle credentials

Same as before: the token is stored as a Colab **secret** named `KAGGLE_API_TOKEN` (key icon in the left bar, *Notebook access* on). Never paste the token into a code cell. If the earlier token was exposed, generate a new one on kaggle.com (*Settings → API*) and update the secret.

If `full.npz` already exists in Drive from an earlier run of this notebook, steps 3–5 are skipped automatically.

In [ ]:
import os
if os.path.exists(f'{OUT}/full.npz'):
    print('data already prepared in Drive: skipping steps 3-5')
else:
    from google.colab import userdata
    token = userdata.get('KAGGLE_API_TOKEN')          # the secret's NAME, not its value
    assert token and token.startswith('KGAT_'), 'Add the KAGGLE_API_TOKEN secret first (key icon, left bar)'
    os.environ['KAGGLE_API_TOKEN'] = token
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/access_token', 'w') as f:
        f.write(token)
    os.chmod('/root/.kaggle/access_token', 0o600)
    print('token set')

## 4. Download the container-flow file only

Downloads `Containers_Dataset.csv` (about 2 GB unzipped) instead of the whole 8.4 GB bundle. If the single-file download fails, it falls back to `kagglehub` for the whole dataset.

In [ ]:
import subprocess, glob, os
DATASET = 'rogernickanaedevha/integrated-cloud-security-3datasets-ics3d'
CSV = '/content/ics3d/Containers_Dataset.csv'
if not os.path.exists(f'{OUT}/full.npz') and not os.path.exists(CSV):
    os.makedirs('/content/ics3d', exist_ok=True)
    r = subprocess.run(['kaggle', 'datasets', 'download', '-d', DATASET, '-f', 'Containers_Dataset.csv',
                        '-p', '/content/ics3d', '--unzip', '-q'], capture_output=True, text=True)
    for z in glob.glob('/content/ics3d/*.zip'):
        subprocess.run(['unzip', '-o', '-q', z, '-d', '/content/ics3d']); os.remove(z)
    if not os.path.exists(CSV):
        print('single-file download failed, using kagglehub:', r.stderr[-300:])
        import kagglehub
        path = kagglehub.dataset_download(DATASET)
        hits = glob.glob(f'{path}/**/Containers_Dataset.csv', recursive=True)
        assert hits, 'Containers_Dataset.csv not found in the download'
        os.symlink(hits[0], CSV)
print(os.path.exists(CSV) and f'{os.path.getsize(CSV)/1e6:.0f} MB  {CSV}')

## 5. Prepare (cap 20,000 flows per class, drop identifiers and the source port, remove duplicates)

In [ ]:
if not os.path.exists(f'{OUT}/full.npz'):
    !python scripts/v2_run.py prepare --csv "{CSV}" --label-col Label --drop-cols "Src Port" --out {OUT} 2>&1 | tee -a {OUT}/run.log
import json
meta = json.load(open(f'{OUT}/meta.json'))
print(json.dumps({k: meta[k] for k in ('classes', 'class_counts_after_dedup', 'dropped_small_classes',
                                      'dropped_label_like_columns', 'dedup', 'time_column')}, indent=1))
print(len(meta['raw_features']), 'raw features')

## 6. Smoke test (about 5–10 minutes)

One seed, one epoch, tiny attacks, in a scratch folder. It only checks that every stage runs on this machine before the long run starts. Its numbers mean nothing.

In [ ]:
TEST = '/content/v2_smoke'
!rm -rf {TEST} && mkdir -p {TEST} && cp {OUT}/full.npz {OUT}/meta.json {TEST}/
!python scripts/v2_run.py train --out {TEST} --seeds 0 --epochs 1 --n-val-robust 200 --models sode_base sode mlp lgbm sode_at mlp_at sode_rs mlp_rs
!python scripts/v2_run.py attack --out {TEST} --seeds 0 --models sode mlp lgbm sode_at mlp_at --n-attack 100 --restarts 1 --long-steps 5 --eot 2 --eot-apgd 2 --apgd-iters 5 --eot-steps 5 --square-queries 20 --em-paths 16 --sanity-seeds
!python scripts/v2_run.py certify --out {TEST} --seeds 0 --models sode_rs mlp_rs --n-cert 50 --smooth-n 200
!python scripts/v2_run.py aggregate --out {TEST} --n-boot 100 | tail -5

## 7. Train the core models on five seeds

Twelve models per seed: `sode_base`, `sode`, `ode`, `mlp`, `lgbm`, `xgb`, `rf`, `sode_at`, `ode_at`, `mlp_at`, `sode_rs`, `mlp_rs`. This is the longest step. If the session disconnects, re-run steps 1–3 and this cell: trained models are skipped.

In [ ]:
!python scripts/v2_run.py train --out {OUT} --seeds {SEEDS} 2>&1 | tee -a {OUT}/run.log | grep -v ' epoch '

## 8. Attacks (lite profile, all seeds)

Per model and seed: clean metrics on the full test split; then PGD-40 and PGD-100 with 2 restarts (fixed paths), APGD-CE and APGD-DLR (EOT 4 for SODE-Guard), EOT-PGD-50 (EOT 8), a 64-path expected-margin attack, Square (1,000 queries) and transfer from the MLP and the adversarially trained MLP, on 500 class-stratified test flows. Trees get Square and transfer only. Seed 0 also runs the ε and step sweeps (gradient-masking checks).

In [ ]:
!python scripts/v2_run.py attack --out {OUT} --seeds {SEEDS} --profile lite 2>&1 | tee -a {OUT}/run.log

## 9. Certificates (randomised smoothing, σ = 0.25, n = 2,000, α = 0.001)

In [ ]:
!python scripts/v2_run.py certify --out {OUT} --seeds {SEEDS} --profile lite 2>&1 | tee -a {OUT}/run.log

## 10. Aggregate over seeds

In [ ]:
!python scripts/v2_run.py aggregate --out {OUT} 2>&1 | tee -a {OUT}/run.log

## 11. Optional checks (run after step 10 if time allows)

**(a) Attack-strength check.** Re-attacks seed 0 with the *full* profile (PGD-200 × 5 restarts, EOT 32, 256-path expected margin, Square 5,000, 1,000 flows) and certifies with n = 10,000, in a separate folder. If the full-profile worst case is close to the lite one, the lite numbers can be reported for all seeds.

In [ ]:
FULL = OUT + '_fullcheck'
!mkdir -p {FULL}/seed_0 && cp {OUT}/full.npz {OUT}/meta.json {FULL}/ && cp {OUT}/seed_0/data.npz {OUT}/seed_0/split.json {OUT}/seed_0/*.pt {OUT}/seed_0/*.pkl {FULL}/seed_0/
!python scripts/v2_run.py attack --out {FULL} --seeds 0 --profile full --models sode sode_at ode_at mlp_at mlp lgbm 2>&1 | tee -a {FULL}/run.log
!python scripts/v2_run.py certify --out {FULL} --seeds 0 --profile full --models sode_rs mlp_rs 2>&1 | tee -a {FULL}/run.log
!python scripts/v2_run.py aggregate --out {FULL} --n-boot 200 2>&1 | tee -a {FULL}/run.log

**(b) Ablations on the same five seeds.** Post-hoc off / per-class, plain CE, focal, class-balanced CE, square-root sampling, quantile input encoding, PGD-5 and TRADES adversarial training, PGD-5 MLP, smoothed neural ODE, SDE-TGNN. Then re-run step 10.

In [ ]:
ABL = 'sode_noposthoc sode_perclass sode_ce sode_focal sode_cb sode_sampler sode_quant sode_at_pgd sode_at_trades mlp_at_pgd ode_rs sdetgnn'
!python scripts/v2_run.py train --out {OUT} --seeds {SEEDS} --models {ABL} 2>&1 | tee -a {OUT}/run.log | grep -v ' epoch '
!python scripts/v2_run.py attack --out {OUT} --seeds {SEEDS} --profile lite --models {ABL} 2>&1 | tee -a {OUT}/run.log
!python scripts/v2_run.py certify --out {OUT} --seeds {SEEDS} --profile lite --models ode_rs 2>&1 | tee -a {OUT}/run.log
!python scripts/v2_run.py aggregate --out {OUT} 2>&1 | tee -a {OUT}/run.log

**(c) Per-class chronological split.** Each class's flows are ordered by timestamp: the first 60% train, the next 20% validate, the last 20% test. This replaces the degenerate global temporal split (whose test window held one class). Three seeds, core models without the trees' extra copies.

In [ ]:
TIME = OUT + '_classtime'
!mkdir -p {TIME} && cp {OUT}/full.npz {OUT}/meta.json {TIME}/
!python scripts/v2_run.py train --out {TIME} --split class_time --seeds 0 1 2 --models sode ode mlp lgbm sode_at mlp_at 2>&1 | tee -a {TIME}/run.log | grep -v ' epoch '
!python scripts/v2_run.py attack --out {TIME} --seeds 0 1 2 --profile lite --models sode ode mlp lgbm sode_at mlp_at --sanity-seeds 2>&1 | tee -a {TIME}/run.log
!python scripts/v2_run.py aggregate --out {TIME} --n-boot 500 2>&1 | tee -a {TIME}/run.log

## 12. Download the results

Zips every `summary.md`, `summary.json`, `meta.json`, `results.json`, `split.json`, the per-flow prediction files and `run.log` (no model weights). Upload `v2_results.zip` to the chat.

In [ ]:
import zipfile, glob, os
from google.colab import files
keep = ('.md', '.json', '.log', '.npz')
with zipfile.ZipFile('/content/v2_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for root in [OUT, OUT + '_fullcheck', OUT + '_classtime']:
        for f in glob.glob(f'{root}/**/*', recursive=True):
            name = os.path.basename(f)
            if f.endswith(keep) and name not in ('full.npz', 'data.npz'):
                z.write(f, os.path.relpath(f, os.path.dirname(OUT)))
print(round(os.path.getsize('/content/v2_results.zip') / 1e6, 1), 'MB')
files.download('/content/v2_results.zip')